# 🩺 Breast Cancer Wisconsin: End-to-End ML Pipeline

**Objective:** Build a robust, reproducible machine learning pipeline for binary classification using tabular data.

**Author:** Amir (Applied AI Engineer Roadmap)

**Date:** 1405/07/25

---

## 📋 Table of Contents
1. [Setup & Data Loading](#1-setup--data-loading)
2. [Exploratory Data Analysis (EDA)](#2-exploratory-data-analysis)
3. [Data Preprocessing & Splitting](#3-data-preprocessing--splitting)
4. [Baseline Model](#4-baseline-model)
5. [Model Training (Logistic Regression)](#5-model-training)
6. [Model Evaluation](#6-model-evaluation)
7. [Error Analysis](#7-error-analysis)
8. [Cross-Validation](#8-cross-validation)

## 1. Setup & Data Loading

In this section, we import the necessary libraries and load the dataset.

In [ ]:
# Import core libraries
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Import scikit-learn modules
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', '{:.2f}'.format)

In [ ]:
# Load the dataset as a pandas DataFrame
dataset = load_breast_cancer(as_frame=True)
df = dataset.frame.copy()

print(f"Dataset shape: {df.shape}")
print(f"Target classes: {dataset.target_names}")
df.head()

## 2. Exploratory Data Analysis

We perform a quick sanity check on the data: dimensions, data types, missing values, and target distribution.

In [ ]:
# Data structure and quality check
print("=== Data Types ===")
print(df.dtypes.value_counts())

print("\n=== Missing Values ===")
print(f"Total missing values: {df.isna().sum().sum()}")

print("\n=== Target Distribution ===")
print(df['target'].value_counts())
print(df['target'].value_counts(normalize=True).round(4) * 100)

## 3. Data Preprocessing & Splitting

We separate features (`X`) from the target (`y`) and split the data into training and test sets. We use `stratify=y` to preserve the class distribution and `random_state=42` for reproducibility.

In [ ]:
# Separate features and target
X = df.drop(columns=['target'])
y = df['target']

# Split the data (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")

## 4. Baseline Model

A `DummyClassifier` that always predicts the majority class provides a simple baseline. Any real model must significantly outperform this.

In [ ]:
# Build and evaluate the baseline
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
baseline_predictions = baseline.predict(X_test)
baseline_accuracy = accuracy_score(y_test, baseline_predictions)

print(f"Baseline Accuracy: {baseline_accuracy:.4f}")

## 5. Model Training

We build a `Pipeline` that combines `StandardScaler` and `LogisticRegression`. This ensures that scaling parameters are learned **only** from the training data, preventing data leakage.

In [ ]:
# Build the pipeline
model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=2000, random_state=42))
])

# Train the model
model.fit(X_train, y_train)
print("Model trained successfully.")

# Generate predictions
predictions = model.predict(X_test)

## 6. Model Evaluation

We calculate the main classification metrics and display the confusion matrix.

In [ ]:
# Calculate metrics
accuracy = accuracy_score(y_test, predictions)
precision = precision_score(y_test, predictions)
recall = recall_score(y_test, predictions)
f1 = f1_score(y_test, predictions)

print("=== Classification Report ===")
print(classification_report(y_test, predictions, target_names=dataset.target_names))

print("=== Confusion Matrix ===")
cm = confusion_matrix(y_test, predictions)
print(cm)

In [ ]:
# Visualize the confusion matrix using a heatmap (optional but recommended for reports)
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=dataset.target_names,
            yticklabels=dataset.target_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

## 7. Error Analysis

We inspect the samples where the model made incorrect predictions to understand its weaknesses.

In [ ]:
# Create a DataFrame for error analysis
error_df = X_test.copy()
error_df['actual'] = y_test
error_df['predicted'] = predictions
errors = error_df[error_df['actual'] != error_df['predicted']]

print(f"Number of errors: {len(errors)}")
print("\nError samples:")
print(errors[['actual', 'predicted']].head())

## 8. Cross-Validation

To ensure the model's performance is stable and not dependent on a single train/test split, we perform 5-fold `StratifiedKFold` cross-validation on the training data.

In [ ]:
# Setup StratifiedKFold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Run cross-validation
cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='f1')

print(f"F1 Scores per fold: {cv_scores}")
print(f"Mean F1 Score: {cv_scores.mean():.4f}")
print(f"Standard Deviation: {cv_scores.std():.4f}")

---
## 📊 Summary of Results

| Metric | Baseline | Logistic Regression |
| :--- | :--- | :--- |
| Accuracy | ~0.63 | **~0.98** |
| F1-Score | ~0.77 | **~0.98** |

**Conclusion:** The `LogisticRegression` model significantly outperforms the baseline, achieving ~98% accuracy. Cross-validation confirms the model's stability.

**Limitations:** This is an educational project. The model is not intended for clinical use.